In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../orders_enriched.csv")
df.head()

,order_id,order_date,customer_id,zip,city,region,district,order_status,payment_method,device_type,order_source,sales_employee_id,sales_employee_name,marital_status,education_level,years_experience,comment
0,1,2012-07-04,58578,1109,Hanoi,East,District #02,delivered,credit_card,desktop,paid_search,EMP0103,Nguyễn Thị Phúc,Đã kết hôn,Đại học,20,"Dịch vụ chăm sóc khách hàng chuyên nghiệp, thá..."
1,2,2012-07-04,58621,1330,Phu Ly,East,District #02,returned,cod,mobile,paid_search,EMP0180,Lê Gia Hùng,Đã kết hôn,Sau đại học,3,"Dịch vụ ổn định, nhân viên luôn sẵn sàng hỗ tr..."
2,3,2012-07-04,58811,1473,Lao Cai,East,District #02,delivered,credit_card,desktop,direct,EMP0093,Bùi Minh Quân,Đã kết hôn,Cao đẳng,18,Khách hàng không có thêm phản hồi về chất lượn...
3,4,2012-07-04,59453,2360,Son Tay,East,District #02,delivered,credit_card,desktop,referral,EMP0015,Võ Quốc Mai,Độc thân,Trung cấp,12,"Phản hồi thắc mắc đầy đủ, hỗ trợ xuyên suốt qu..."
4,6,2012-07-06,57821,2886,Uong Bi,East,District #02,delivered,paypal,mobile,email_campaign,EMP0107,Hồ Thanh Yến,Độc thân,Đại học,17,Khách hàng đánh giá mức độ hài lòng ở mức khá.


In [3]:
# Xem kiểu dữ liệu và có giá trị null không
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 17 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   order_id             646945 non-null  int64 
 1   order_date           646945 non-null  object
 2   customer_id          646945 non-null  int64 
 3   zip                  646945 non-null  int64 
 4   city                 646945 non-null  object
 5   region               646945 non-null  object
 6   district             646945 non-null  object
 7   order_status         646945 non-null  object
 8   payment_method       646945 non-null  object
 9   device_type          646945 non-null  object
 10  order_source         646945 non-null  object
 11  sales_employee_id    646945 non-null  object
 12  sales_employee_name  646945 non-null  object
 13  marital_status       646945 non-null  object
 14  education_level      646945 non-null  object
 15  years_experience     646945 non-nu

In [4]:
# Drop cột thông tin không cần thiết vì có thể join thông qua khoá ngoại.
# Các trường như sales_employee_name, marital_status, education_level, 
# years_experience sẽ tạo ra một bảng riêng.
employees = df[[
    'sales_employee_id', 'sales_employee_name',
    'marital_status', 'education_level', 'years_experience'
]].drop_duplicates()
df = df.drop(columns=['city', 'region', 'district', 'payment_method', 'sales_employee_name', 'marital_status', 'education_level', 'years_experience'], axis=1)
display(employees.head())
employees.info()


,sales_employee_id,sales_employee_name,marital_status,education_level,years_experience
0,EMP0103,Nguyễn Thị Phúc,Đã kết hôn,Đại học,20
1,EMP0180,Lê Gia Hùng,Đã kết hôn,Sau đại học,3
2,EMP0093,Bùi Minh Quân,Đã kết hôn,Cao đẳng,18
3,EMP0015,Võ Quốc Mai,Độc thân,Trung cấp,12
4,EMP0107,Hồ Thanh Yến,Độc thân,Đại học,17


<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, 0 to 1076
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   sales_employee_id    200 non-null    object
 1   sales_employee_name  200 non-null    object
 2   marital_status       200 non-null    object
 3   education_level      200 non-null    object
 4   years_experience     200 non-null    int64 
dtypes: int64(1), object(4)
memory usage: 9.4+ KB


In [5]:
for col in employees.select_dtypes(include="object").columns:
    employees[col] = employees[col].str.strip()
employees = employees.replace("", pd.NA)
employees.isna().sum()


sales_employee_id      0
sales_employee_name    0
marital_status         0
education_level        0
years_experience       0
dtype: int64

In [6]:
# Kiểm tra thử có bao nhiêu giá trị trong marital_status, education_level, years_experience
display(employees["marital_status"].value_counts(dropna=False))
display(employees["education_level"].value_counts(dropna=False))
display(employees["years_experience"].value_counts(dropna=False).sort_index())

marital_status
Đã kết hôn    107
Độc thân       93
Name: count, dtype: int64

education_level
Đại học        96
Cao đẳng       44
Sau đại học    30
Trung cấp      30
Name: count, dtype: int64

years_experience
1      7
2      6
3     13
4      9
5      7
6      6
7      9
8     10
9     14
10    16
11     9
12    14
13     9
14    11
15     8
16     9
17    14
18    11
19    10
20     8
Name: count, dtype: int64

In [7]:
employees.to_csv("../SilverData/employees_silver.csv", index=False)

In [8]:
# Kiểm tra số lượng null trong các cột
df.isna().sum()

order_id             0
order_date           0
customer_id          0
zip                  0
order_status         0
device_type          0
order_source         0
sales_employee_id    0
comment              0
dtype: int64

In [9]:
# Kiểm tra thử có order_id trùng lặp không
print("order_id is unique     :", df["order_id"].is_unique)

order_id is unique     : True


In [10]:
# Xem thử có bao nhiêu giá trị trong order_status
df["order_status"].value_counts(dropna=False)

order_status
delivered    516716
cancelled     59462
returned      36142
shipped       13773
paid          13577
created        7275
Name: count, dtype: int64

In [11]:
# Xem thử có bao nhiêu giá trị trong device_type
df["device_type"].value_counts(dropna=False)

device_type
mobile     291482
desktop    258855
tablet      96608
Name: count, dtype: int64

In [12]:
# Xem thử có bao nhiêu giá trị trong order_source
df["order_source"].value_counts(dropna=False)

order_source
organic_search    181495
paid_search       141652
social_media      129710
email_campaign     77572
referral           64565
direct             51951
Name: count, dtype: int64

In [13]:
# Chuẩn hóa khoảng trắng dư thừa cho các cột dạng chuỗi
str_cols = df.select_dtypes(include=["object"]).columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip()

In [14]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
print("Số ngày không đọc được:", df["order_date"].isna().sum())

df["order_id"] = df["order_id"].astype("int32")
df["customer_id"] = df["customer_id"].astype("int32")
df["zip"] = df["zip"].astype("int32")

cat_cols = ["order_status", "device_type", "order_source", "sales_employee_id"]
for col in cat_cols:
    df[col] = df[col].astype("category")

df.info()

Số ngày không đọc được: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   order_id           646945 non-null  int32         
 1   order_date         646945 non-null  datetime64[ns]
 2   customer_id        646945 non-null  int32         
 3   zip                646945 non-null  int32         
 4   order_status       646945 non-null  category      
 5   device_type        646945 non-null  category      
 6   order_source       646945 non-null  category      
 7   sales_employee_id  646945 non-null  category      
 8   comment            646945 non-null  object        
dtypes: category(4), datetime64[ns](1), int32(3), object(1)
memory usage: 20.4+ MB


In [15]:
# Sắp xếp theo order_id và reset index
df = df.sort_values("order_id").reset_index(drop=True)

print("Tổng số dòng     :", len(df))
print("Tổng số null     :", df.isna().sum().sum())

Tổng số dòng     : 646945
Tổng số null     : 0


In [16]:
df.to_csv("../SilverData/orders_silver.csv", index=False)